# Conversion Heatmaps

This notebook visualizes the summarized conversion results from `results/summary/conversion_by_pair.csv`.
Regenerate the summaries first if needed.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))


            from pathlib import Path

            import matplotlib.pyplot as plt
            import pandas as pd

            RESULTS_PATH = ROOT / "results" / "summary" / "conversion_by_pair.csv"
            assert RESULTS_PATH.exists(), f"Missing summary CSV: {RESULTS_PATH}"
            conversion_df = pd.read_csv(RESULTS_PATH)
            conversion_df.head()


In [ ]:
def expand_rows(frame):
    rows = []
    for _, record in frame.iterrows():
        rows.append(
            {
                "direction": f"{record['source_modality']}→{record['target_modality']}",
                "source_subject": record["source_subject"],
                "target_subject": record["target_subject"],
                "top1": record["forward_top1"],
                "top5": record["forward_top5"],
                "two_way": record["forward_two_way"],
            }
        )
        rows.append(
            {
                "direction": f"{record['target_modality']}→{record['source_modality']}",
                "source_subject": record["target_subject"],
                "target_subject": record["source_subject"],
                "top1": record["reverse_top1"],
                "top5": record["reverse_top5"],
                "two_way": record["reverse_two_way"],
            }
        )
    return pd.DataFrame(rows)

long_df = expand_rows(conversion_df)
long_df.head()


In [ ]:
DIRECTION = sorted(long_df["direction"].unique())[0]
METRIC = "top1"

subset = long_df[long_df["direction"] == DIRECTION]
heatmap = subset.pivot(index="source_subject", columns="target_subject", values=METRIC)

fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(heatmap.values, cmap="YlGnBu")
ax.set_title(f"{DIRECTION} | {METRIC}")
ax.set_xlabel("Target subject")
ax.set_ylabel("Source subject")
ax.set_xticks(range(len(heatmap.columns)))
ax.set_xticklabels(heatmap.columns)
ax.set_yticks(range(len(heatmap.index)))
ax.set_yticklabels(heatmap.index)
fig.colorbar(image, ax=ax)
plt.show()
